# Session 5 · Demonstrating MapReduce Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-05-mapreduce-algorithms.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · A reusable MapReduce simulator with a combiner

In [2]:
from collections import defaultdict

def run_mapreduce(splits, mapper, reducer, combiner=None, n_reducers=2, partitioner=None):
    partitioner = partitioner or (lambda k, r: sum(map(ord, str(k))) % r)
    shuffled = [defaultdict(list) for _ in range(n_reducers)]
    sent = 0
    for split in splits:                                   # one mapper per split
        local = defaultdict(list)
        for record in split:
            for k, v in mapper(record):
                local[k].append(v)
        if combiner:
            local = {k: [combiner(k, vs)] for k, vs in local.items()}
        for k, vs in local.items():
            sent += len(vs)
            shuffled[partitioner(k, n_reducers)][k].extend(vs)
    outputs = [[reducer(k, vs) for k, vs in sorted(part.items())] for part in shuffled]
    return outputs, sent

splits = [["the cat sat on the mat", "the dog sat"], ["the cat ran", "a dog and the cat"]]
mapper = lambda line: ((w, 1) for w in line.split())
reducer = lambda k, vs: (k, sum(vs))

for comb in [None, reducer]:
    out, sent = run_mapreduce(splits, mapper, reducer, combiner=(lambda k, vs: sum(vs)) if comb else None)
    print(f"combiner={'yes' if comb else 'no ':<3}  pairs sent over network = {sent}")
for i, part in enumerate(out):
    print(f"part-r-0000{i}: {part}")

combiner=no   pairs sent over network = 17
combiner=yes  pairs sent over network = 12
part-r-00000: [('cat', 3), ('dog', 2), ('mat', 1), ('sat', 2)]
part-r-00001: [('a', 1), ('and', 1), ('on', 1), ('ran', 1), ('the', 5)]


### Example 2 · Search: distributed grep (map-only)

In [3]:
import re
logs = {
    "node1/blk_1": ["2026-10-01 10:02 INFO login user=ali", "2026-10-01 10:05 ERROR db timeout"],
    "node2/blk_2": ["2026-10-01 11:00 WARN slow query", "2026-10-01 11:07 ERROR disk full on dn4"],
    "node3/blk_3": ["2026-10-02 09:15 INFO login user=sara", "2026-10-02 09:20 ERROR db timeout"],
}
pattern = re.compile(r"ERROR (.*)")

def grep_mapper(location, offset, line):
    m = pattern.search(line)
    if m:
        yield f"{location}:{offset}", m.group(1)

for loc, lines in logs.items():                   # each block scanned by its own mapper
    for off, line in enumerate(lines):
        for key, value in grep_mapper(loc, off, line):
            print(f"{key:<14} {value}")

node1/blk_1:1  db timeout
node2/blk_2:1  disk full on dn4
node3/blk_3:1  db timeout


### Example 3 · Total-order sort with a range partitioner

In [4]:
import random
random.seed(1)
records = [(random.randint(1, 999), f"student{i:02d}") for i in range(18)]   # (score, name)

sample = sorted(k for k, _ in random.sample(records, 6))     # sample keys to pick split points
cuts = [sample[2], sample[4]]                                # 3 reducers -> 2 split points
def range_partition(key):
    return 0 if key < cuts[0] else 1 if key < cuts[1] else 2

parts = {0: [], 1: [], 2: []}
for k, v in records:
    parts[range_partition(k)].append((k, v))
print("split points:", cuts)
for r in parts:
    parts[r].sort()                                          # the shuffle sorts each reducer's input
    print(f"part-r-0000{r}: {[k for k, _ in parts[r]]}")
merged = [k for r in parts for k, _ in parts[r]]
print("concatenated output is globally sorted:", merged == sorted(merged))

split points:

 [215, 668]
part-r-00000: [65, 97, 121, 138]
part-r-00001: [215, 262, 389, 461, 484, 500, 508, 583]
part-r-00002: [668, 780, 783, 808, 822, 868]
concatenated output is globally sorted: True


### Example 4 · Building an inverted index

In [5]:
from collections import defaultdict
docs = {
    "d1": "hadoop stores big data",
    "d2": "spark processes big data in memory",
    "d3": "hive runs sql on hadoop",
}
# map
pairs = [(w, d) for d, text in docs.items() for w in set(text.split())]
# shuffle + reduce
index = defaultdict(set)
for w, d in pairs:
    index[w].add(d)
for w in sorted(index):
    print(f"{w:<10} -> {sorted(index[w])}")

query = ["big", "hadoop"]
hits = set.intersection(*(index[w] for w in query))
print(f"\nquery {query} -> {sorted(hits)}")

big        -> ['d1', 'd2']
data       -> ['d1', 'd2']
hadoop     -> ['d1', 'd3']
hive       -> ['d3']
in         -> ['d2']
memory     -> ['d2']
on         -> ['d3']
processes  -> ['d2']
runs       -> ['d3']
spark      -> ['d2']
sql        -> ['d3']
stores     -> ['d1']

query ['big', 'hadoop'] -> ['d1']


### Example 5 · TF-IDF as three chained MapReduce jobs

In [6]:
import math
from collections import defaultdict
docs = {"d1": "big data big clusters", "d2": "big data in hdfs", "d3": "spark streams data fast"}
N = len(docs)

# Job 1: ((word, doc), n)
j1 = defaultdict(int)
for d, text in docs.items():
    for w in text.split():
        j1[(w, d)] += 1
# Job 2: key by doc to get total words per doc
per_doc = defaultdict(list)
for (w, d), n in j1.items():
    per_doc[d].append((w, n))
j2 = {(w, d): (n, sum(c for _, c in wl)) for d, wl in per_doc.items() for w, n in wl}
# Job 3: key by word; the number of values = document frequency
per_word = defaultdict(list)
for (w, d), (n, total) in j2.items():
    per_word[w].append((d, n, total))
scores = {(w, d): (n / total) * math.log(N / len(vals))
          for w, vals in per_word.items() for d, n, total in vals}
for (w, d), s in sorted(scores.items(), key=lambda x: (x[0][1], -x[1])):
    print(f"{d}  {w:<9} {s:.3f}")

d1  clusters  0.275
d1  big       0.203
d1  data      0.000
d2  in        0.275
d2  hdfs      0.275
d2  big       0.101
d2  data      0.000
d3  spark     0.275
d3  streams   0.275
d3  fast      0.275
d3  data      0.000


### Example 6 · The same TF-IDF in Spark (real PySpark)

In [7]:
import math
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]").appName("tfidf")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

docs = sc.parallelize([("d1", "big data big clusters"), ("d2", "big data in hdfs"),
                       ("d3", "spark streams data fast")])
N = docs.count()
tf = (docs.flatMap(lambda d: [((w, d[0]), 1) for w in d[1].split()])
          .reduceByKey(lambda a, b: a + b)                                   # job 1
          .map(lambda x: (x[0][1], (x[0][0], x[1]))))
doc_len = docs.map(lambda d: (d[0], len(d[1].split())))
tf = tf.join(doc_len).map(lambda x: (x[1][0][0], (x[0], x[1][0][1] / x[1][1])))  # job 2
df = tf.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
tfidf = tf.join(df).map(lambda x: (x[1][0][0], x[0], round(x[1][0][1] * math.log(N / x[1][1]), 3)))  # job 3
for d, w, s in sorted(tfidf.collect(), key=lambda r: (r[0], -r[2], r[1])):
    print(d, f"{w:<9}", s)

d1 clusters  0.275
d1 big       0.203
d1 data      0.0
d2 hdfs      0.275
d2 in        0.275
d2 big       0.101
d2 data      0.0
d3 fast      0.275
d3 spark     0.275
d3 streams   0.275
d3 data      0.0
